# 반도체 공정 데이터 EDA 연습

`반도체_공정_샘플.csv`(200행)를 이용하여 수업에서 배운 EDA 과정을 처음부터 끝까지 연습합니다.

**불러오기 → 살펴보기 → 정리하기(결측·중복) → 분석하기(필터·groupby·그래프)**

> 이 노트북은 `반도체_공정_샘플.csv`와 **같은 폴더**에서 실행하세요.

## 0. 준비 - 라이브러리와 데이터 불러오기

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 그래프 한글 설정 (Mac은 'AppleGothic')
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

df = pd.read_csv('반도체_공정_샘플.csv', encoding='utf-8')
df.head()

# Part 1. 데이터 살펴보기

## 1. 데이터 크기 확인 - `shape`

In [ ]:
print(f"데이터 크기: {df.shape[0]}행 × {df.shape[1]}열")

## 2. 위에서 5줄 / 아래에서 3줄 보기 - `head`, `tail`

In [ ]:
display(df.head())
display(df.tail(3))

## 3. 열 이름, 타입, 결측 수 확인 - `info`

In [ ]:
df.info()

## 4. 기술 통계 확인 - `describe`

In [ ]:
df.describe().round(2)

## 5. 원하는 열 선택

In [ ]:
# 한 개 열 → Series
display(df['온도_섭씨'].head())

# 여러 열 → DataFrame (대괄호 두 겹)
df[['온도_섭씨', '압력_Pa', '합격여부']].head()

## 6. 원하는 행 선택 - `iloc`, `loc`

In [ ]:
display(df.iloc[0])           # 첫 번째 행
display(df.iloc[1:5])         # 1~4번 행 (5번 미포함)
df.loc[0:4, ['온도_섭씨', '합격여부']]   # 0~4번 행 (4번 포함)

# Part 2. 합격여부 살펴보기

## 7. 합격/불합격 개수 - `value_counts`

In [ ]:
df['합격여부'].value_counts()

## 8. 합격/불합격 비율(%) - `normalize=True`

In [ ]:
df['합격여부'].value_counts(normalize=True) * 100

# Part 3. 결측값과 중복 처리

## 9. 열별 결측값 개수 - `isnull().sum()`

In [ ]:
print(df.isnull().sum())
print(f"전체 결측값 개수: {df.isnull().sum().sum()}개")

## 10. 평균으로 결측값 채우기 - `fillna`

In [ ]:
df_filled = df.copy()   # 원본 보호를 위해 복사본 사용

numeric_cols = ['온도_섭씨', '압력_Pa', '가스유량_slm', '전력_W',
                '진공도_mTorr', '두께_nm', '습도_pct', '진동_mm_s',
                '처리시간_sec', '냉각수온도_섭씨']

for col in numeric_cols:
    df_filled[col] = df_filled[col].fillna(df_filled[col].mean())

print(f"처리 후 남은 결측값: {df_filled.isnull().sum().sum()}개")

## 11. 중복 행 확인과 제거 - `duplicated`, `drop_duplicates`

In [ ]:
print(f"중복 행 수: {df_filled.duplicated().sum()}개")

df_clean = df_filled.drop_duplicates().reset_index(drop=True)
print(f"정리 전: {len(df_filled)}행 → 정리 후: {len(df_clean)}행")

# Part 4. 새 컬럼 만들기와 조건 필터링

## 12. 합격여부를 '합격'/'불합격' 글자로 바꾸기 - `map`

In [ ]:
df_clean['result'] = df_clean['합격여부'].map({1: '합격', -1: '불합격'})
df_clean[['온도_섭씨', '두께_nm', '합격여부', 'result']].head()

## 13. 불합격 데이터만 골라내기

In [ ]:
fail_df = df_clean[df_clean['합격여부'] == -1]
print(f"불합격 건수: {len(fail_df)}건")
fail_df.head()

## 14. 온도가 305도를 넘는 공정

In [ ]:
high_temp_df = df_clean[df_clean['온도_섭씨'] > 305]
print(f"고온 공정 수: {len(high_temp_df)}건")

## 15. 여러 조건 동시에 - 고온 AND 두꺼운 박막

In [ ]:
condition = (df_clean['온도_섭씨'] > 305) & (df_clean['두께_nm'] > 103)
risk_df = df_clean[condition]
print(f"고온+두꺼움 공정: {len(risk_df)}건")
risk_df[['온도_섭씨', '두께_nm', 'result']].head()

# Part 5. 그룹별 비교와 그래프

## 16. 합격/불합격별 평균 비교 - `groupby`

In [ ]:
group_mean = df_clean.groupby('result')[numeric_cols].mean().round(2)
group_mean.T

## 17. 히스토그램 - 온도 분포

In [ ]:
plt.hist(df_clean['온도_섭씨'], bins=20, edgecolor='white')
plt.title('온도 분포')
plt.show()

## 18. 박스플롯 - 합격/불합격별 온도 비교

In [ ]:
df_clean.boxplot(column='온도_섭씨', by='result')
plt.show()

# 분석 정리

위 결과를 보고 아래 빈칸을 직접 채워 보세요.

1. 합격 ___건 / 불합격 ___건 → 불합격 비율 ___%
2. 불합격 공정은 합격보다 평균 온도가 약 ___°C (높다/낮다)
3. 불합격 공정은 두께가 약 ___nm (두껍다/얇다)

# 종합 연습문제

아래 문제의 정답 코드를 직접 작성해 보세요. (`df_clean`은 위에서 만든 정리된 데이터입니다.)

1. `shape`을 사용하여 데이터의 행 수와 열 수를 출력하세요.
2. 열별 결측값 개수 중 **가장 많은 열의 이름**과 개수를 구하세요. (원본 `df` 사용)
3. `df_clean`에서 합격 건수와 불합격 건수를 `value_counts`로 출력하세요.
4. `df_clean`에서 `전력_W`가 2050 이상인 공정의 건수를 구하세요.
5. `loc`을 사용하여 `압력_Pa`가 1020 이상인 행의 `측정시간`, `압력_Pa`, `result`를 출력하세요.
6. `groupby`로 판정별 `전력_W` 평균을 구하고 두 값의 차이를 출력하세요.
7. `진공도_mTorr`를 판정별 박스플롯으로 그리세요.
8. `냉각수온도_섭씨` 히스토그램을 `bins=15`로 그리세요.
9. `습도_pct`가 50 이상이면서 `온도_섭씨`가 300 이상인 공정의 건수와, 그중 불합격 건수를 구하세요.
10. `iloc`을 사용하여 처음 3개 행, 처음 4개 열을 출력하세요.

# 연습문제 정답

아래 셀은 연습문제를 모두 풀어본 후 확인하세요.

### 연습문제 1 정답

In [ ]:
print(df.shape)

### 연습문제 2 정답

In [ ]:
na = df.isnull().sum()
print(na.idxmax(), na.max())

### 연습문제 3 정답

In [ ]:
df_clean['result'].value_counts()

### 연습문제 4 정답

In [ ]:
print(len(df_clean[df_clean['전력_W'] >= 2050]))

### 연습문제 5 정답

In [ ]:
df_clean.loc[df_clean['압력_Pa'] >= 1020, ['측정시간', '압력_Pa', 'result']]

### 연습문제 6 정답

In [ ]:
m = df_clean.groupby('result')['전력_W'].mean()
print(m)
print(f"차이: {m['불합격'] - m['합격']:.2f}")

### 연습문제 7 정답

In [ ]:
df_clean.boxplot(column='진공도_mTorr', by='result')
plt.show()

### 연습문제 8 정답

In [ ]:
plt.hist(df_clean['냉각수온도_섭씨'], bins=15, edgecolor='white')
plt.title('냉각수온도 분포')
plt.show()

### 연습문제 9 정답

In [ ]:
c = (df_clean['습도_pct'] >= 50) & (df_clean['온도_섭씨'] >= 300)
sub = df_clean[c]
print(len(sub), len(sub[sub['합격여부'] == -1]))

### 연습문제 10 정답

In [ ]:
df_clean.iloc[0:3, 0:4]